# Pipeline Preprocessing Teks (Bulan 2)
Notebook ini menggabungkan tahapan **Basic Cleaning (Minggu 5-6)** dan **Advanced Normalization (Minggu 7-8)** untuk keperluan Evaluasi Tengah Semester (UTS).

## 1. Persiapan & Memuat Data Mentah
Kita memuat dataset `Gabungan.xlsx` yang ada di folder Tubes/.

In [ ]:
import pandas as pd
import re

# Sumber data utama (Tubes/Gabungan.xlsx)
# Kalo datanya di-update, tinggal re-run notebook ini aja
MASTER_DATA = '../Gabungan.xlsx'

df = pd.read_excel(MASTER_DATA)
print(f'Jumlah data mentah (awal): {len(df)} baris')
df[['full_text']].head()

## 2. Minggu 5-6 (Basic Cleaning)
Tahap membersihkan data yang kotor menggunakan **Case Folding**, **Regex Cleaning**, dan **Filtering**.

In [ ]:
def basic_cleaning(text):
    if not isinstance(text, str):
        return ''
    
    # A. Case folding (mengecilkan huruf)
    text = text.lower()
    
    # B. Regex cleaning
    text = re.sub(r'http\S+|www\.\S+', '', text) # Hapus URL
    text = re.sub(r'#\w+', '', text)            # Hapus hashtag
    text = re.sub(r'@\w+', '', text)            # Hapus mention
    text = re.sub(r'\d+', '', text)             # Hapus angka
    
    text = re.sub(r'[^\w\s]', ' ', text)       # Hapus tanda baca/spesial karakter
    text = re.sub(r'\s+', ' ', text).strip()    # Hapus spasi ganda
    
    return text

# Mengaplikasikan pembersihan ke dataset
df['cleaned_text'] = df['full_text'].apply(basic_cleaning)

# C. Filtering (menghapus tweet yang terlalu pendek / < 3 kata)
df['word_count'] = df['cleaned_text'].apply(lambda x: len(str(x).split()))
df_filtered = df[df['word_count'] >= 3].copy()
df_filtered = df_filtered.drop(columns=['word_count'])

print(f'Jumlah data setelah filtering (bersih): {len(df_filtered)} baris')
print(f'Data noise yang dibuang: {len(df) - len(df_filtered)} baris')

## 3. Minggu 7-8 (Advanced Normalization)
Tahap krusial untuk menormalisasi kata gaul (slang) menjadi kata formal agar lebih mudah dianalisis oleh model selanjutnya.

In [ ]:
# Membangun kamus pemetaan (Slang Dictionary)
slang_dict = {
    'aq': 'saya', 'aku': 'saya', 'gue': 'saya', 'gw': 'saya',
    'bngung': 'bingung', 'bgt': 'sekali', 'banget': 'sekali',
    'mo': 'mau', 'yg': 'yang', 'dgn': 'dengan', 'utk': 'untuk',
    'jd': 'jadi', 'tp': 'tapi', 'kalo': 'kalau', 'kl': 'kalau',
    'klo': 'kalau', 'aja': 'saja', 'jgn': 'jangan', 'udh': 'sudah',
    'udah': 'sudah', 'gak': 'tidak', 'ga': 'tidak', 'g': 'tidak',
    'bisa': 'bisa', 'laper': 'lapar', 'healing': 'penyembuhan',
    'stress': 'stres', 'bikin': 'membuat', 'bener': 'benar',
    'tau': 'tahu', 'taunya': 'tahunya', 'kayak': 'seperti',
    'kyk': 'seperti', 'emang': 'memang', 'krn': 'karena',
    'karna': 'karena', 'knetz': 'netizen korea', 'indo': 'indonesia'
}

def normalize_slang(text, dictionary):
    if not isinstance(text, str):
        return text
    words = text.split()
    normalized_words = [dictionary.get(word, word) for word in words]
    return ' '.join(normalized_words)

In [ ]:
# Uji coba fungsi (contoh spesifik)
contoh_teks = 'aq bngung bgt mo nangis'
print('Data Bersih  :', contoh_teks)
print('Data Normal  :', normalize_slang(contoh_teks, slang_dict))

In [ ]:
# Mengaplikasikan normalisasi pada dataset yang sudah dibersihkan
df_filtered['normalized_text'] = df_filtered['cleaned_text'].apply(lambda x: normalize_slang(x, slang_dict))

## 4. UTS (Evaluasi I): Demo Perbandingan Data Mentah vs Data Bersih (Data Integrity)
Menampilkan perbandingan data awal vs hasil akhir untuk memastikan bahwa proses cleaning dan normalisasi tidak mengubah struktur makna (Data Integrity).

In [ ]:
pd.set_option('display.max_colwidth', None)
demo_df = df_filtered[['full_text', 'cleaned_text', 'normalized_text']].sample(10, random_state=42)

demo_df.rename(columns={
    'full_text': 'Teks Asli (Kotor)', 
    'cleaned_text': 'Basic Cleaned (Minggu 5-6)', 
    'normalized_text': 'Normalized (Minggu 7-8)'
}, inplace=True)

demo_df

In [ ]:
# Menyimpan dataset gabungan hasil pipeline yang sudah final
df_filtered.to_excel('pipeline_akhir_bulan_2.xlsx', index=False)
print(f'Proses Selesai! {len(df_filtered)} baris data disimpan di: pipeline_akhir_bulan_2.xlsx')